# 02 — Carver trend system vs ensembles

Two philosophies, same BTC book, same costs, `exec_lag=1`.

* **Ensemble (QMIE spot analog):** binary flag, full-port when on, flat when off. Times turns; lumpy DD.
* **Carver:** continuous forecast → vol-targeted size. Always allocated at some (possibly tiny) weight. Surfs the trend.
* **Blend:** 50/50 unlagged mix, then lagged once in the backtest. Diversifies timing vs sizing.

The vol target is the **master dial**. Raise it and both return and DD scale; the *shape* of the curve stays the same. That is the prop-firm use case in the source note — QMIE still does not send orders.

## Hypotheses

| Id | Claim |
|---|---|
| H5 | Carver has lower OOS DD (and usually lower CAGR) than the binary ensemble |
| H6 | ADX chop gate and/or a causal DD circuit breaker tighten OOS max DD vs raw Carver |


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "research").exists():
    pass
elif (ROOT / "python" / "research").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root", ROOT)


In [ ]:
from research.trend_lab.allocation import blend_weights, chop_gate
from research.trend_lab.carver import backtest, dd_circuit_breaker, full_carver
from research.trend_lab.data import CORE, load_panel, load_symbol
from research.trend_lab.metrics import kpis
from research.trend_lab.plots import allocation_fig, equity_overlay, rolling_sharpe_fig, underwater
from research.trend_lab.protocol import WARMUP_BARS, split_frame
from research.trend_lab.spot_system import SpotParams, spot_signal
import pandas as pd

btc, _ = load_symbol("BTCUSDT", "1d")
parts = split_frame(btc)
panel, srcs = load_panel(CORE[:4], "1d")
print("panel", list(panel.columns), srcs)
w, fc, fdm = full_carver(panel, "BTCUSDT", use_cs=panel.shape[1] >= 3)
print("FDM", round(fdm, 3))
cv = backtest(btc["close"], w.reindex(btc.index).fillna(0.0))
ens = spot_signal(btc, SpotParams())
mix = backtest(btc["close"], blend_weights(w.reindex(btc.index).fillna(0.0), ens["signal"], mix=0.5))
gate = chop_gate(btc, 18.0)
chop = backtest(btc["close"], w.reindex(btc.index).fillna(0.0) * gate.reindex(btc.index).fillna(0.0))
brk = backtest(btc["close"], dd_circuit_breaker(w.reindex(btc.index).fillna(0.0), cv["equity"]))

def oos(bt):
    sl = bt.reindex(parts["oos"].index)
    return kpis(sl["net"], sl["equity"])

rows = {
    "ensemble_OOS": kpis(ens.reindex(parts["oos"].index)["net"], ens.reindex(parts["oos"].index)["equity"]),
    "carver_OOS": oos(cv),
    "blend_OOS": oos(mix),
    "chop_carver_OOS": oos(chop),
    "dd_breaker_OOS": oos(brk),
}
display(pd.DataFrame(rows).T.round(3))


In [ ]:
vol_rows = []
for vt in (0.10, 0.20, 0.40):
    wv, _, _ = full_carver(panel, "BTCUSDT", use_cs=panel.shape[1] >= 3, vol_target=vt)
    bt = backtest(btc["close"], wv.reindex(btc.index).fillna(0.0)).reindex(parts["oos"].index)
    vol_rows.append({"vol_target": vt, **kpis(bt["net"], bt["equity"])})
display(pd.DataFrame(vol_rows).round(3))

oos_eq = {
    "ensemble": ens.reindex(parts["oos"].index)["equity"],
    "carver": cv.reindex(parts["oos"].index)["equity"],
    "blend": mix.reindex(parts["oos"].index)["equity"],
    "chop": chop.reindex(parts["oos"].index)["equity"],
    "dd-breaker": brk.reindex(parts["oos"].index)["equity"],
}
equity_overlay(oos_eq, "OOS growth — Carver vs ensemble").show()
rolling_sharpe_fig({k: v.pct_change().fillna(0) for k, v in oos_eq.items()}, 90, "OOS 90d rolling Sharpe").show()
underwater(cv.reindex(parts["oos"].index)["equity"], "Carver OOS DD").show()
allocation_fig({
    "carver": cv["held"].reindex(parts["oos"].index),
    "ensemble": ens["held"].reindex(parts["oos"].index),
    "blend": mix["held"].reindex(parts["oos"].index),
}, "OOS allocation (lagged weights)").show()


## How to read this

If Carver’s OOS CAGR looks “emasculating” next to the ensemble, that is the product, not a bug: vol targeting sells headline return for a smoother path. The ensemble will usually win **timing** on a single name when the flag is well fitted. Carver wins **mandate fit** when a daily-loss cap exists.

Trend following still needs a trend. Sideways OOS will flatten both books; the chop gate is allowed to stay flat. Do not engineer that away by fitting ADX on OOS.
